# Multi-Agent Workflow for Biomedical Literature Review
## AI Engineering Case Study

**Architecture:** Three-agent pipeline over PubMed Central Open Access articles.

| Agent | Role | Model |
|---|---|---|
| Retriever | Semantic search over article corpus | `sentence-transformers/all-MiniLM-L6-v2` |
| Summarizer | Abstractive summarization of abstracts | `facebook/bart-large-cnn` |
| Verifier | Factual consistency checking via NLI | `facebook/bart-large-mnli` |

**Data source:** PMC OA Commercial subset (`pmc-oa-opendata`, S3, us-east-1, anonymous access)

**Implementation notes:**
- S3 paths use `deprecated/` prefix (active until August 2026; original paths moved April 2026)
- `facebook/bart-large-cnn` chosen over `facebook/bart-base`: the base model is not fine-tuned for summarization and produces incoherent output; bart-large-cnn is fine-tuned on CNN/DailyMail and produces coherent abstractive summaries
- All models are CPU-compatible; no cloud credentials required
- Total model download on first run: approximately 3.5 GB

---

## Section 1: Setup and Data Access

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────────
import codecs
import csv
import json
import re
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from datetime import datetime

# ── Third-party ───────────────────────────────────────────────────────────────
import boto3
import numpy as np
import pandas as pd
from botocore import UNSIGNED
from botocore.client import Config
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# ── Configuration ─────────────────────────────────────────────────────────────
PMC_BUCKET    = "pmc-oa-opendata"

# Paths use the deprecated/ prefix added April 2026; valid until August 2026.
# The assignment PDF shows the pre-migration paths which no longer exist.
FILELIST_KEY  = "deprecated/oa_comm/xml/metadata/csv/oa_comm.filelist.csv"
XML_PREFIX    = "deprecated/oa_comm/xml/all/"

MAX_ARTICLES  = 100
TOP_K         = 5
RANDOM_SEED   = 42

SUMMARIZER_MODEL = "facebook/bart-large-cnn"
VERIFIER_MODEL   = "facebook/bart-large-mnli"
EMBEDDER_MODEL   = "sentence-transformers/all-MiniLM-L6-v2"

TEST_QUERIES = [
    "Adverse events with mRNA vaccines in pediatrics",
    "Transformer-based models for protein folding",
    "Clinical trial outcomes for monoclonal antibodies in oncology",
]

print("Configuration")
print(f"  Embedder model  : {EMBEDDER_MODEL}")
print(f"  Summarizer model: {SUMMARIZER_MODEL}")
print(f"  Verifier model  : {VERIFIER_MODEL}")
print(f"  Max articles    : {MAX_ARTICLES}")
print(f"  Top-K per query : {TOP_K}")
print("OK")

Configuration
  Embedder model  : sentence-transformers/all-MiniLM-L6-v2
  Summarizer model: facebook/bart-large-cnn
  Verifier model  : facebook/bart-large-mnli
  Max articles    : 100
  Top-K per query : 5
OK


In [ ]:
# PMC S3 bucket is public. Anonymous (unsigned) requests are required;
# signed requests are rejected even with valid credentials.
s3 = boto3.client(
    "s3",
    region_name="us-east-1",
    config=Config(signature_version=UNSIGNED),
)

print("S3 client configured (anonymous, us-east-1)")

try:
    s3.head_bucket(Bucket=PMC_BUCKET)
    print(f"Bucket '{PMC_BUCKET}' is accessible")
except Exception as e:
    print(f"Bucket check failed: {e}")
    print("Verify network connectivity and that us-east-1 is reachable.")

S3 client configured (anonymous, us-east-1)
Bucket 'pmc-oa-opendata' is accessible


In [ ]:
@dataclass
class Article:
    """
    Represents a single PMC article with metadata and agent-populated fields.

    Fields pmcid through keywords are parsed from JATS XML.
    Fields embedding through verification_score are populated by agents at runtime.
    """
    # Parsed from JATS XML
    pmcid:        str
    pmid:         str
    doi:          str
    title:        str
    abstract:     str
    authors:      str
    journal:      str
    pub_year:     str
    article_type: str
    keywords:     str

    # Populated by RetrieverAgent
    embedding:    np.ndarray | None = field(default=None, repr=False)
    relevance:    float                = 0.0

    # Populated by SummarizerAgent
    summary:            str  = ""
    extracted_keywords: list = field(default_factory=list)

    # Populated by VerifierAgent
    verified_summary:   str   = ""
    verification_score: float = 0.0
    verification_flag:  str   = ""   # "passed" | "fallback" | "error"

    def text_for_embedding(self) -> str:
        """Title-injected text used for embedding. Improves hit@k over abstract-only."""
        return f"Title: {self.title}. {self.abstract}".strip()

    def __repr__(self) -> str:
        return (
            f"Article(pmcid={self.pmcid!r}, "
            f"title={self.title[:55]!r}..., "
            f"relevance={self.relevance:.3f})"
        )

print("Article dataclass defined")

Article dataclass defined


In [ ]:
def parse_jats_xml(xml_bytes: bytes) -> dict | None:
    """
    Parse a JATS XML article and extract key metadata fields.

    PMC articles use JATS (Journal Article Tag Suite) XML.
    Handles both structured abstracts (sections with <sec> elements)
    and flat abstracts (plain <p> children).

    Returns None when no usable abstract is found; such articles are
    quarantined at fetch time and excluded from the corpus.
    """
    # Strip UTF-8 BOM if present
    if xml_bytes[:3] == b"\xef\xbb\xbf":
        xml_bytes = xml_bytes[3:]

    try:
        root = ET.fromstring(xml_bytes)
    except ET.ParseError:
        return None

    def find_id(pub_type: str) -> str:
        for el in root.findall(f".//article-id[@pub-id-type='{pub_type}']"):
            return (el.text or "").strip()
        return ""

    def all_text(el) -> str:
        return "".join(el.itertext()).strip()

    pmcid = find_id("pmc") or find_id("pmcid")
    pmid  = find_id("pmid")
    doi   = find_id("doi")

    title_el = root.find(".//article-title")
    title    = all_text(title_el) if title_el is not None else ""

    # Abstract: prefer structured (Background/Methods/Results) over flat
    abstract_parts = []
    for abs_el in root.findall(".//abstract"):
        secs = abs_el.findall(".//sec")
        if secs:
            for sec in secs:
                sec_title_el = sec.find("title")
                sec_title    = all_text(sec_title_el) if sec_title_el is not None else ""
                sec_body     = " ".join(all_text(p) for p in sec.findall(".//p"))
                if sec_body:
                    prefix = f"{sec_title}: " if sec_title else ""
                    abstract_parts.append(f"{prefix}{sec_body}")
        else:
            body = " ".join(all_text(p) for p in abs_el.findall(".//p"))
            if not body:
                body = all_text(abs_el)
            if body:
                abstract_parts.append(body)

    abstract = " ".join(abstract_parts).strip()
    if not abstract:
        return None  # quarantine: no abstract

    # Authors: cap at 5 to keep the field concise
    authors = []
    for contrib in root.findall(".//contrib[@contrib-type='author']")[:5]:
        surname_el = contrib.find("name/surname")
        given_el   = contrib.find("name/given-names")
        surname    = all_text(surname_el) if surname_el is not None else ""
        given      = all_text(given_el)   if given_el   is not None else ""
        initial    = given[:1] if given else ""
        if surname:
            authors.append(f"{surname} {initial}".strip())

    # Publication year: try named date types in preference order
    pub_year = ""
    for date_type in ("epub", "ppub", "pmc-release", "collection"):
        for pd_el in root.findall(f".//pub-date[@date-type='{date_type}']"):
            yr = (pd_el.findtext("year") or "").strip()
            if yr:
                pub_year = yr
                break
        if pub_year:
            break
    if not pub_year:
        for pd_el in root.findall(".//pub-date"):
            yr = (pd_el.findtext("year") or "").strip()
            if yr:
                pub_year = yr
                break

    journal      = (root.findtext(".//journal-title") or "").strip()
    article_type = root.attrib.get("article-type", "")
    keywords     = ", ".join(
        (kw.text or "").strip()
        for kw in root.findall(".//kwd-group/kwd")
        if kw.text
    )[:200]

    return dict(
        pmcid=pmcid, pmid=pmid, doi=doi,
        title=title, abstract=abstract,
        authors=", ".join(authors),
        journal=journal, pub_year=pub_year,
        article_type=article_type, keywords=keywords,
    )

print("JATS XML parser defined")

JATS XML parser defined


In [ ]:
def load_filelist(max_articles: int = MAX_ARTICLES) -> pd.DataFrame:
    """
    Read the PMC OA commercial filelist CSV from S3.

    The file is streamed; only the first max_articles*3 XML rows are retained
    to allow for articles that will be quarantined due to missing abstracts.

    CSV columns (as of 2026):
        Key, ETag, Article Citation, AccessionID,
        Last Updated UTC, PMID, License, Retracted
    """
    print(f"Reading filelist: s3://{PMC_BUCKET}/{FILELIST_KEY}")

    obj    = s3.get_object(Bucket=PMC_BUCKET, Key=FILELIST_KEY)
    reader = csv.DictReader(codecs.getreader("utf-8")(obj["Body"]))

    rows = []
    for row in reader:
        if row.get("Retracted", "").strip().lower() in ("yes", "true"):
            continue
        if not row.get("Key", "").strip().endswith(".xml"):
            continue
        rows.append(row)
        if len(rows) >= max_articles * 3:
            break

    df = pd.DataFrame(rows)
    print(f"Loaded {len(df)} candidate rows")
    print(f"Columns: {list(df.columns)}")
    return df


filelist_df = load_filelist()
filelist_df.head(3)

Reading filelist: s3://pmc-oa-opendata/deprecated/oa_comm/xml/metadata/csv/oa_comm.filelist.csv


Loaded 300 candidate rows
Columns: ['Key', 'ETag', 'Article Citation', 'AccessionID', 'Last Updated UTC (YYYY-MM-DD HH:MM:SS)', 'PMID', 'License', 'Retracted']


,Key,ETag,Article Citation,AccessionID,Last Updated UTC (YYYY-MM-DD HH:MM:SS),PMID,License,Retracted
0,deprecated/oa_comm/xml/all/PMC10001703.xml,aee58fb7603de552a789ab5fe5c6a682,Phys Med Biol. 2023 Mar 21; 68(6):065003,PMC10001703,2026-05-14 23:37:16,36780696,CC BY,no
1,deprecated/oa_comm/xml/all/PMC10004261.xml,3eac27eaeedb879223fc29f58fb0706d,Adv Drug Alcohol Res. 2022 Dec 5; 2:10831,PMC10004261,2026-04-22 23:42:22,36908580,CC BY,no
2,deprecated/oa_comm/xml/all/PMC10006208.xml,a7e3f444a489a1c20059a4d53b3d7030,NPJ Regen Med. 2023 Mar 10; 8:14,PMC10006208,2026-05-30 23:36:41,36899012,CC BY,no


In [ ]:
def fetch_articles(
    filelist_df: pd.DataFrame,
    max_articles: int = MAX_ARTICLES,
) -> list[Article]:
    """
    Fetch and parse XML articles from S3.

    Two-layer retraction filtering is applied:
        1. Filelist Retracted column (catches most cases at CSV read time)
        2. Title-level guard: rejects articles whose title starts with
           RETRACTED, WITHDRAWN, CORRECTION, or ERRATUM. This catches
           cases where the filelist column is stale or incorrectly populated.

    Articles without an abstract are also quarantined.
    Network or parse errors are counted as quarantined.
    """
    articles    = []
    quarantined = 0
    attempted   = 0

    for _, row in filelist_df.iterrows():
        if len(articles) >= max_articles:
            break

        xml_key = row.get("Key", "").strip()
        if not xml_key:
            continue

        attempted += 1
        try:
            obj      = s3.get_object(Bucket=PMC_BUCKET, Key=xml_key)
            xml_data = obj["Body"].read()
            parsed   = parse_jats_xml(xml_data)

            if parsed is None:
                quarantined += 1
                continue

            # Secondary retraction guard: the filelist Retracted column is
            # sometimes stale. Reject any article whose title begins with
            # RETRACTED, WITHDRAWN, or CORRECTION to prevent surfacing
            # invalidated work in a clinical literature review.
            title_upper = parsed.get("title", "").upper().strip()
            if any(title_upper.startswith(marker) for marker in
                   ("RETRACTED", "WITHDRAWN", "CORRECTION", "ERRATUM")):
                print(f"  Quarantined (retracted title): {parsed['title'][:60]}")
                quarantined += 1
                continue

            # Backfill PMID from filelist when not present in the XML
            if not parsed["pmid"] and row.get("PMID"):
                parsed["pmid"] = str(row["PMID"]).strip()

            articles.append(Article(**parsed))

        except Exception as e:
            print(f"  Skipped {xml_key}: {type(e).__name__}")
            quarantined += 1

    print(f"Fetched {len(articles)} articles with abstracts")
    print(f"Attempted: {attempted}  |  Quarantined: {quarantined}")
    return articles


articles = fetch_articles(filelist_df)

print()
print("Sample article")
print(f"  PMID     : {articles[0].pmid}")
print(f"  Title    : {articles[0].title[:80]}")
print(f"  Abstract : {articles[0].abstract[:200]}...")

  Quarantined (retracted title): RETRACTED: Molecular Investigation of DKK3 in Cerebral Ische
  Quarantined (retracted title): RETRACTED: LncRNA ERVH48-1 Contributes to the Drug Resistanc


  Quarantined (retracted title): RETRACTED ARTICLE: Cerebrolysin Attenuates Exacerbation of N


  Quarantined (retracted title): RETRACTED ARTICLE: Double-negative metamaterial square enclo


Fetched 100 articles with abstracts
Attempted: 109  |  Quarantined: 9

Sample article
  PMID     : 36780696
  Title    : Stereotactic body radiation therapy (SBRT) following Yttrium-90 (90Y) selective 
  Abstract : Objective. 90Y selective internal radiation therapy (SIRT) treatment of hepatocellular carcinoma (HCC) can potentially underdose lesions, as identified on post-therapy PET/CT imaging. This study intro...


---
## Section 2: Retriever Agent

**Task:** Given a natural-language research query, return the top-K most semantically relevant articles from the corpus.

**Design choices:**
- Model: `sentence-transformers/all-MiniLM-L6-v2` (384-dim, 256 token limit, ~90 MB)
- Similarity: cosine similarity between normalised query and article embeddings
- Corpus indexed once at startup; each query requires only one forward pass
- Title injection: title is prepended to abstract before embedding, which improves hit@k because query terms frequently match article titles before dense abstract text
- Optional multi-query expansion: runs original and synonym-expanded queries, merges results, deduplicates by PMID

**Tradeoff:** Pure semantic retrieval; no BM25 keyword fallback. For 100 articles this is adequate. At production scale, hybrid BM25 + semantic search improves precision on exact drug names and gene symbols.

In [ ]:
print(f"Loading {EMBEDDER_MODEL} ...")
print("First run downloads approximately 90 MB from HuggingFace.")

embedder = SentenceTransformer(EMBEDDER_MODEL)

test_emb = embedder.encode(["test"], convert_to_numpy=True)
assert test_emb.shape[1] == 384, f"Unexpected embedding dim: {test_emb.shape[1]}"
print(f"Embedding model loaded. Output dimension: {test_emb.shape[1]} (expected 384)")

Loading sentence-transformers/all-MiniLM-L6-v2 ...
First run downloads approximately 90 MB from HuggingFace.


Embedding model loaded. Output dimension: 384 (expected 384)


In [ ]:
def build_index(
    articles: list[Article],
    embedder: SentenceTransformer,
) -> np.ndarray:
    """
    Encode all articles in a single batched call and attach embeddings.

    normalize_embeddings=True normalises vectors to unit length so that
    cosine similarity equals dot product, which avoids a separate
    normalisation step and is slightly faster.

    Returns corpus_embeddings of shape (n_articles, 384).
    """
    texts = [a.text_for_embedding() for a in articles]

    print(f"Encoding {len(texts)} articles ...")
    embeddings = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    for article, emb in zip(articles, embeddings):
        article.embedding = emb

    print(f"Index built: shape {embeddings.shape}")
    return embeddings


corpus_embeddings = build_index(articles, embedder)

Encoding 100 articles ...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Index built: shape (100, 384)


In [ ]:
class RetrieverAgent:
    """
    Retriever Agent: semantic search over the PMC article corpus.

    Accepts a natural-language query and returns the top-K articles
    ranked by cosine similarity between query and article embeddings.

    Output is passed directly to SummarizerAgent.
    """

    # Synonym expansions for common biomedical query terms.
    # Uses natural-language phrasing rather than Boolean OR tokens,
    # which would degrade embedding quality by introducing literal 'OR'.
    EXPANSIONS = {
        "adverse events":      "adverse events, side effects, safety profile, tolerability",
        "pediatric":           "pediatric, children, adolescent, paediatric",
        "mRNA vaccine":        "mRNA vaccine, mRNA vaccination, COVID-19 vaccine, BNT162b2",
        "protein folding":     "protein folding, protein structure prediction, AlphaFold",
        "transformer":         "transformer, attention mechanism, BERT, deep learning model",
        "monoclonal antibody": "monoclonal antibody, biologic therapy, targeted therapy, mAb",
        "oncology":            "oncology, cancer, tumor, neoplasm, malignancy",
        "clinical trial":      "clinical trial, randomized controlled trial, RCT",
    }

    def __init__(
        self,
        articles: list[Article],
        corpus_embeddings: np.ndarray,
        embedder: SentenceTransformer,
        top_k: int = TOP_K,
    ):
        self.articles          = articles
        self.corpus_embeddings = corpus_embeddings
        self.embedder          = embedder
        self.top_k             = top_k

    def retrieve(self, query: str, top_k: int | None = None) -> list[Article]:
        """
        Embed query, compute cosine similarity against corpus,
        return top-K articles sorted by descending relevance score.
        """
        k = top_k or self.top_k

        query_emb = self.embedder.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
        similarities = cosine_similarity(query_emb, self.corpus_embeddings)[0]
        top_indices  = np.argsort(similarities)[::-1][:k]

        results = []
        for idx in top_indices:
            art           = self.articles[idx]
            art.relevance = float(similarities[idx])
            results.append(art)
        return results

    def retrieve_with_expansion(self, query: str, top_k: int | None = None) -> list[Article]:
        """
        Run the original query and a synonym-expanded variant, then merge
        and deduplicate results by PMID, keeping the highest relevance score.
        Use this when the standard retrieve() returns low-scoring results.
        """
        k            = top_k or self.top_k
        expanded     = self._expand_query(query)
        results_orig = self.retrieve(query,    top_k=k)
        results_exp  = self.retrieve(expanded, top_k=k)

        seen: dict[str, Article] = {}
        for art in results_orig + results_exp:
            key = art.pmid or art.pmcid
            if key not in seen or art.relevance > seen[key].relevance:
                seen[key] = art

        return sorted(seen.values(), key=lambda a: a.relevance, reverse=True)[:k]

    def _expand_query(self, query: str) -> str:
        expanded = query
        for term, expansion in self.EXPANSIONS.items():
            if term.lower() in query.lower():
                expanded = expanded.replace(term, expansion)
        return expanded

    def __repr__(self) -> str:
        return (
            f"RetrieverAgent("
            f"n_articles={len(self.articles)}, "
            f"embedding_dim={self.corpus_embeddings.shape[1]}, "
            f"top_k={self.top_k})"
        )


retriever = RetrieverAgent(articles, corpus_embeddings, embedder, top_k=TOP_K)
print(retriever)

RetrieverAgent(n_articles=100, embedding_dim=384, top_k=5)


In [ ]:
# Smoke test: run all three case study queries
for query in TEST_QUERIES:
    print(f"\n{'=' * 60}")
    print(f"Query: {query}")
    print(f"{'=' * 60}")
    for rank, art in enumerate(retriever.retrieve(query), 1):
        print(f"  [{rank}] score={art.relevance:.3f}  PMID:{art.pmid or 'N/A'}")
        print(f"       {art.title[:72]}")
        print(f"       {art.journal} ({art.pub_year})")

print("\nRetriever agent OK")


Query: Adverse events with mRNA vaccines in pediatrics
  [1] score=0.339  PMID:36756808
       The Pregnancy, Arsenic, and Immune Response (PAIR) Study in rural northe
       Paediatric and Perinatal Epidemiology (2023)
  [2] score=0.292  PMID:36755464
       Rare and low-frequency coding genetic variants contribute to pediatric-o
       Multiple Sclerosis (Houndmills, Basingstoke, England) (2023)
  [3] score=0.274  PMID:35801814
       Pediatric surgical site infections in 287 hospitals in the United States
       Infection Control and Hospital Epidemiology (2023)
  [4] score=0.249  PMID:36883013
       Comparative transcriptomics from intestinal cells of permissive and non-
       Parasitology (2023)
  [5] score=0.238  PMID:33860505
       Evaluation of the Seroprevalence of Infectious Diseases in 2,445 in vitr
       Revista Brasileira de Ginecologia e Obstetrícia (2021)

Query: Transformer-based models for protein folding
  [1] score=0.478  PMID:37020023
       De novo design of m

---
## Section 3: Summarizer Agent

**Task:** Produce a 2-3 sentence abstractive summary and extract keywords from each retrieved article.

**Design choices:**
- Model: `facebook/bart-large-cnn` (~1.6 GB, fine-tuned on CNN/DailyMail)
- Chosen over `facebook/bart-base`: the base model is not fine-tuned for summarization (its HuggingFace task tag is `feature-extraction`); calling it via the summarization pipeline produces incoherent output
- `sshleifer/distilbart-cnn-6-6` would be a lighter alternative but does not have safetensors weights, which are required on torch 2.5+ due to CVE-2025-32434
- Keyword extraction: frequency-based over title + abstract; no additional model dependency
- Study type classification: rule-based matching on abstract text

**Tradeoff:** distilbart-cnn-6-6 is not domain-adapted to biomedical text. For production, a model such as `allenai/led-base-16384` or `microsoft/BioGPT` would improve summary quality on dense scientific abstracts.

In [ ]:
print(f"Loading {SUMMARIZER_MODEL} ...")
print("First run downloads approximately 1.6 GB from HuggingFace.")

summarizer_pipeline = pipeline(
    "summarization",
    model=SUMMARIZER_MODEL,
    device=-1,       # -1 = CPU; set to 0 for GPU
    truncation=True,
)

_test = summarizer_pipeline(
    "Deep learning has shown strong results in biomedical NLP tasks including "
    "named entity recognition, relation extraction, and clinical text classification.",
    max_length=40, min_length=15, do_sample=False,
)
print(f"Summarizer loaded. Test output: {_test[0]['summary_text']}")

Loading facebook/bart-large-cnn ...
First run downloads approximately 1.6 GB from HuggingFace.


Device set to use cpu


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Your max_length is set to 40, but your input_length is only 26. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)


Summarizer loaded. Test output: Deep learning has shown strong results in biomedical NLP tasks including named entity recognition, relation extraction, and clinical text classification.


In [ ]:
def extract_keywords(text: str, n: int = 5) -> list[str]:
    """
    Extract top-N keywords by term frequency.

    Filters English stopwords and tokens shorter than 4 characters.
    No additional model or library is required.
    """
    STOPWORDS = {
        "the", "a", "an", "and", "or", "but", "in", "on", "at", "to",
        "for", "of", "with", "by", "from", "is", "are", "was", "were",
        "be", "been", "have", "has", "had", "do", "does", "did", "will",
        "would", "could", "should", "may", "might", "this", "that", "these",
        "those", "it", "its", "we", "our", "they", "their", "which", "who",
        "as", "than", "more", "less", "also", "not", "no", "can", "all",
        "both", "each", "between", "results", "study", "patients", "patient",
        "significantly", "however", "while", "using", "used", "based",
    }
    tokens = re.findall(r"\b[a-zA-Z]{4,}\b", text.lower())
    tokens = [t for t in tokens if t not in STOPWORDS]
    freq   = {}
    for t in tokens:
        freq[t] = freq.get(t, 0) + 1
    return sorted(freq, key=freq.get, reverse=True)[:n]


def classify_study_type(text: str) -> str:
    """
    Rule-based study type classification from abstract text.

    Categories (in priority order):
        systematic review | meta-analysis | RCT | observational |
        case study | computational | review | other
    """
    t = text.lower()
    rules = [
        (["systematic review", "systematic literature"],              "systematic review"),
        (["meta-analysis", "meta analysis"],                          "meta-analysis"),
        (["randomized controlled", "randomised controlled",
          "rct", "double-blind", "placebo-controlled"],               "RCT"),
        (["cohort study", "case-control", "cross-sectional",
          "retrospective", "prospective study"],                       "observational"),
        (["case report", "case series"],                              "case study"),
        (["deep learning", "neural network", "transformer",
          "machine learning", "algorithm", "computational",
          "alphafold", "in silico"],                                  "computational"),
        (["review"],                                                  "review"),
    ]
    for keywords, label in rules:
        if any(kw in t for kw in keywords):
            return label
    return "other"


# Spot check
print("Keyword sample:", extract_keywords(articles[0].abstract))
print("Study type sample:")
for art in articles[:3]:
    print(f"  [{classify_study_type(art.abstract):20s}] {art.title[:60]}")

Keyword sample: ['sirt', 'sbrt', 'plans', 'post', 'tumor']
Study type sample:
  [observational       ] Stereotactic body radiation therapy (SBRT) following Yttrium
  [other               ] Hippocampal ceRNA networks from chronic intermittent ethanol
  [other               ] Parishin A-loaded mesoporous silica nanoparticles modulate m


In [ ]:
class SummarizerAgent:
    """
    Summarizer Agent: abstractive summarization of retrieved article abstracts.

    Receives Article objects from RetrieverAgent and populates:
        article.summary              -- abstractive summary (2-3 sentences)
        article.extracted_keywords   -- top-5 frequency-based keywords

    Output is passed to VerifierAgent for factual consistency checking.
    """

    def __init__(
        self,
        summarizer_pipeline,
        max_summary_tokens: int = 80,
        min_summary_tokens: int = 30,
    ):
        self.pipeline           = summarizer_pipeline
        self.max_summary_tokens = max_summary_tokens
        self.min_summary_tokens = min_summary_tokens

    def summarize_article(self, article: Article) -> Article:
        """
        Generate a summary for a single article.
        Abstract is truncated to 1 500 characters before passing to the model
        to stay within the 1 024 token limit of bart-large-cnn.
        Falls back to the first two sentences of the abstract on error.
        """
        text = article.abstract[:1500]
        try:
            out = self.pipeline(
                text,
                max_length=self.max_summary_tokens,
                min_length=self.min_summary_tokens,
                do_sample=False,
                truncation=True,
            )
            article.summary = out[0]["summary_text"]
        except Exception:
            sentences = re.split(r"(?<=[.!?])\s+", text)
            article.summary = " ".join(sentences[:2])

        article.extracted_keywords = extract_keywords(
            f"{article.title} {article.abstract}", n=5
        )
        return article

    def summarize_batch(self, articles: list[Article]) -> list[Article]:
        """Summarize a list of articles, printing progress."""
        total = len(articles)
        print(f"Summarizing {total} articles ...")
        for i, art in enumerate(articles, 1):
            self.summarize_article(art)
            print(f"  [{i}/{total}] PMID:{art.pmid or 'N/A'} OK")
        return articles

    def __repr__(self) -> str:
        return (
            f"SummarizerAgent(model={SUMMARIZER_MODEL!r}, "
            f"max_tokens={self.max_summary_tokens})"
        )


summarizer_agent = SummarizerAgent(summarizer_pipeline)
print(summarizer_agent)

SummarizerAgent(model='facebook/bart-large-cnn', max_tokens=80)


---
## Section 4: Verifier Agent

**Task:** Check factual consistency of generated summaries against their source abstracts using natural language inference (NLI).

**Design choices:**
- Model: `facebook/bart-large-mnli` -- BART fine-tuned on Multi-Genre NLI (~1.6 GB)
- The zero-shot classification pipeline takes the abstract as the sequence and the summary as the hypothesis
- The entailment score represents the model's confidence that the summary is factually supported by the abstract
- Summaries below the threshold (default 0.5) are replaced with the first two sentences of the source abstract as a conservative fallback
- All verification decisions are recorded on the Article object for transparency in the report

**Why NLI for verification:**
NLI directly models the entailment relationship between a premise and a hypothesis. Treating the abstract as the premise and the summary as the hypothesis gives a principled, model-grounded consistency score, which is more reliable than heuristic overlap metrics (ROUGE, BLEU) for detecting factual errors in biomedical text.

**Tradeoff:** bart-large-mnli is trained on general-domain text. Domain-adapted NLI models (e.g., `microsoft/BiomedNLI`) would improve precision on medical claims, but are not required for this prototype scope.

In [ ]:
print(f"Loading {VERIFIER_MODEL} ...")
print("First run downloads approximately 1.6 GB from HuggingFace.")

# zero-shot-classification is the correct pipeline for BART-MNLI.
# It takes a 'sequence' (premise) and a list of 'candidate_labels' (hypotheses)
# and returns per-label scores derived from the NLI head.
nli_pipeline = pipeline(
    "zero-shot-classification",
    model=VERIFIER_MODEL,
    device=-1,
)

# Sanity check
_premise    = "The vaccine was well tolerated with only mild adverse events reported."
_hypothesis = "The vaccine was safe."
_result     = nli_pipeline(_premise, candidate_labels=[_hypothesis])
print("Verifier loaded.")
print(f"  Sanity check score: {_result['scores'][0]:.3f} (expect > 0.5 for entailment)")

Loading facebook/bart-large-mnli ...
First run downloads approximately 1.6 GB from HuggingFace.


Device set to use cpu


Verifier loaded.
  Sanity check score: 0.983 (expect > 0.5 for entailment)


In [ ]:
class VerifierAgent:
    """
    Verifier Agent: NLI-based factual consistency checking.

    Uses zero-shot-classification with facebook/bart-large-mnli.
    Treats article abstract as the premise and generated summary as the hypothesis.
    The entailment score is the model's confidence that the summary is
    factually supported by the abstract.

    Populates on each Article:
        verified_summary    -- accepted summary or conservative abstract fallback
        verification_score  -- entailment score [0, 1]
        verification_flag   -- 'passed' | 'fallback' | 'error'
    """

    def __init__(self, nli_pipeline, entailment_threshold: float = 0.5):
        self.pipeline   = nli_pipeline
        self.threshold  = entailment_threshold

    def verify_article(self, article: Article) -> Article:
        """
        Verify one article's summary against its abstract.
        Modifies article in-place and returns it.
        """
        if not article.summary or not article.abstract:
            article.verified_summary   = article.summary
            article.verification_score = 0.0
            article.verification_flag  = "error"
            return article

        try:
            result = self.pipeline(
                article.abstract,
                candidate_labels=[article.summary],
            )
            score = float(result["scores"][0])
            article.verification_score = score

            if score >= self.threshold:
                article.verified_summary  = article.summary
                article.verification_flag = "passed"
            else:
                # Conservative fallback: first two sentences of source abstract
                sentences = re.split(r"(?<=[.!?])\s+", article.abstract)
                article.verified_summary  = " ".join(sentences[:2])
                article.verification_flag = "fallback"

        except Exception:
            article.verified_summary   = article.summary
            article.verification_score = 0.0
            article.verification_flag  = "error"

        return article

    def verify_batch(self, articles: list[Article]) -> list[Article]:
        """Verify a batch of articles, printing progress and a summary."""
        total = len(articles)
        print(f"Verifying {total} articles ...")
        for i, art in enumerate(articles, 1):
            self.verify_article(art)
            print(
                f"  [{i}/{total}] PMID:{art.pmid or 'N/A':10} "
                f"score={art.verification_score:.3f}  [{art.verification_flag}]"
            )
        passed   = sum(1 for a in articles if a.verification_flag == "passed")
        fallback = sum(1 for a in articles if a.verification_flag == "fallback")
        errors   = sum(1 for a in articles if a.verification_flag == "error")
        print(f"Verification complete: {passed} passed, {fallback} fallback, {errors} errors")
        return articles

    def __repr__(self) -> str:
        return (
            f"VerifierAgent(model={VERIFIER_MODEL!r}, "
            f"threshold={self.threshold})"
        )


verifier_agent = VerifierAgent(nli_pipeline)
print(verifier_agent)

VerifierAgent(model='facebook/bart-large-mnli', threshold=0.5)


---
## Section 5: Report Generation

Runs the full three-agent pipeline for all three test queries and generates:
1. Structured per-query output (Python dict and flattened DataFrame)
2. Formatted Markdown report (`literature_review_report.md`)
3. Structured JSON output (`pipeline_results.json`)

The report is formatted for an R&D scientist: one section per query, each article as a structured block with verified summary, study type, verification score, keywords, and full citation.

In [ ]:
def run_pipeline(
    query: str,
    retriever: RetrieverAgent,
    summarizer: SummarizerAgent,
    verifier: VerifierAgent,
    top_k: int = TOP_K,
    use_expansion: bool = False,
) -> dict:
    """
    Execute the full three-agent pipeline for a single query.

    Steps:
        1. RetrieverAgent  -- retrieve top-K articles by semantic similarity
        2. SummarizerAgent -- generate abstractive summary + keywords per article
        3. VerifierAgent   -- check factual consistency of each summary

    Returns a structured result dict suitable for report generation.
    """
    # Step 1: Retrieve
    if use_expansion:
        retrieved = retriever.retrieve_with_expansion(query, top_k=top_k)
    else:
        retrieved = retriever.retrieve(query, top_k=top_k)

    # Step 2: Summarize
    summarized = summarizer.summarize_batch(retrieved)

    # Step 3: Verify
    verified = verifier.verify_batch(summarized)

    articles_out = []
    for art in verified:
        articles_out.append({
            "pmid":               art.pmid,
            "pmcid":              art.pmcid,
            "doi":                art.doi,
            "title":              art.title,
            "authors":            art.authors,
            "journal":            art.journal,
            "pub_year":           art.pub_year,
            "study_type":         classify_study_type(art.abstract),
            "relevance_score":    round(art.relevance, 4),
            "summary":            art.verified_summary,
            "verification_score": round(art.verification_score, 4),
            "verification_flag":  art.verification_flag,
            "keywords":           art.extracted_keywords,
            "abstract_snippet":   art.abstract[:300] + "...",
        })

    return {
        "query":              query,
        "run_timestamp":      datetime.utcnow().isoformat() + "Z",
        "model_embedder":     EMBEDDER_MODEL,
        "model_summarizer":   SUMMARIZER_MODEL,
        "model_verifier":     VERIFIER_MODEL,
        "articles_retrieved": len(articles_out),
        "articles":           articles_out,
    }


print("Pipeline function defined")

Pipeline function defined


In [ ]:
all_results: dict = {}

for query in TEST_QUERIES:
    print(f"\n{'=' * 60}")
    print(f"Running pipeline for query: {query}")
    print(f"{'=' * 60}")
    result = run_pipeline(
        query,
        retriever=retriever,
        summarizer=summarizer_agent,
        verifier=verifier_agent,
        top_k=TOP_K,
    )
    all_results[query] = result
    print(f"Query complete: {result['articles_retrieved']} articles retrieved, summarized, and verified")

print("\nAll three queries complete")


Running pipeline for query: Adverse events with mRNA vaccines in pediatrics
Summarizing 5 articles ...


  [1/5] PMID:36756808 OK


Your max_length is set to 80, but your input_length is only 74. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=37)


  [2/5] PMID:36755464 OK


  [3/5] PMID:35801814 OK


...

  [1/5] PMID:36656137   score=0.994  [passed]


  [2/5] PMID:36780696   score=1.000  [passed]


  [3/5] PMID:36752780   score=0.999  [passed]


  [4/5] PMID:36165348   score=0.000  [fallback]


  [5/5] PMID:33860505   score=0.999  [passed]
Verification complete: 4 passed, 1 fallback, 0 errors
Query complete: 5 articles retrieved, summarized, and verified

All three queries complete


In [ ]:
from IPython.display import Markdown, display

rows = []
for query, result in all_results.items():
    for art in result["articles"]:
        rows.append({
            "Query":              query[:45] + "...",
            "PMID":               art["pmid"],
            "Title":              art["title"][:55] + "...",
            "Study type":         art["study_type"],
            "Relevance":          art["relevance_score"],
            "Verif. score":       art["verification_score"],
            "Verif. flag":        art["verification_flag"],
            "Year":               art["pub_year"],
        })

df_results = pd.DataFrame(rows)
display(df_results)

,Query,PMID,Title,Study type,Relevance,Verif. score,Verif. flag,Year
0,Adverse events with mRNA vaccines in pediatri...,36756808,"The Pregnancy, Arsenic, and Immune Response (P...",other,0.3392,0.9969,passed,2023
1,Adverse events with mRNA vaccines in pediatri...,36755464,Rare and low-frequency coding genetic variants...,other,0.2921,0.9986,passed,2023
2,Adverse events with mRNA vaccines in pediatri...,35801814,Pediatric surgical site infections in 287 hosp...,other,0.2741,0.9994,passed,2023
3,Adverse events with mRNA vaccines in pediatri...,36883013,Comparative transcriptomics from intestinal ce...,other,0.2489,0.9960,passed,2023
4,Adverse events with mRNA vaccines in pediatri...,33860505,Evaluation of the Seroprevalence of Infectious...,observational,0.2384,0.9993,passed,2021
5,Transformer-based models for protein folding...,37020023,De novo design of modular peptide-binding prot...,computational,0.4776,0.9993,passed,2023
6,Transformer-based models for protein folding...,37079924,Role of Strong\nLocalized vs Weak Distributed ...,other,0.4703,0.9961,passed,2023
7,Transformer-based models for protein folding...,36759579,A molecular switch modulates assembly and host...,other,0.4676,0.9996,passed,2023
8,Transformer-based models for protein folding...,36692197,Molecular Modeling\nStudy of a Receptor–Orthos...,other,0.3633,0.9993,passed,2023
9,Transformer-based models for protein folding...,36970896,Cathepsin B Deficiency Improves Memory Deficit...,other,0.2316,0.9968,passed,2023


In [ ]:
def generate_markdown_report(all_results: dict, n_corpus_articles: int) -> str:
    """
    Generate a structured Markdown literature review report.

    Format:
        - Header with run metadata
        - One section per query
        - Per article: citation table, verified summary, keywords, verification status
        - Notes section explaining scoring and classification methods
    """
    ts    = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    lines = []

    lines += [
        "# Biomedical Literature Review Report",
        "",
        f"**Generated:** {ts}  ",
        "**Data source:** PubMed Central Open Access (oa_comm)  ",
        f"**Corpus size:** {n_corpus_articles} articles  ",
        f"**Retrieval model:** `{EMBEDDER_MODEL}`  ",
        f"**Summarization model:** `{SUMMARIZER_MODEL}`  ",
        f"**Verification model:** `{VERIFIER_MODEL}`  ",
        "",
        "---",
        "",
    ]

    for query, result in all_results.items():
        lines += [
            f"## Query: {query}",
            "",
            f"*{result['articles_retrieved']} articles retrieved*",
            "",
        ]

        for rank, art in enumerate(result["articles"], 1):
            pmid_md = (
                f"[PMID:{art['pmid']}](https://pubmed.ncbi.nlm.nih.gov/{art['pmid']}/)"
                if art["pmid"] else "N/A"
            )
            doi_md = (
                f"[{art['doi']}](https://doi.org/{art['doi']})"
                if art["doi"] else "N/A"
            )
            verif_badge = (
                "Passed" if art["verification_flag"] == "passed"
                else "Fallback" if art["verification_flag"] == "fallback"
                else "Error"
            )

            lines += [
                f"### {rank}. {art['title']}",
                "",
                "| Field | Value |",
                "|---|---|",
                f"| Citation | {pmid_md} |",
                f"| DOI | {doi_md} |",
                f"| Authors | {art['authors'] or 'N/A'} |",
                f"| Journal | {art['journal'] or 'N/A'} ({art['pub_year'] or 'N/A'}) |",
                f"| Study type | `{art['study_type']}` |",
                f"| Retrieval score | {art['relevance_score']:.4f} |",
                f"| Verification score | {art['verification_score']:.4f} ({verif_badge}) |",
                "",
                f"**Summary:** {art['summary']}",
                "",
                f"**Keywords:** {', '.join(f'`{k}`' for k in art['keywords'])}",
                "",
            ]

        lines += ["---", ""]

    lines += [
        "## Notes",
        "",
        "- **Retrieval score:** cosine similarity between query and article embeddings (range 0-1; higher is more relevant).",
        "- **Verification score:** NLI entailment score indicating how well the summary is supported by the source abstract (range 0-1).",
        "- **Fallback:** summaries below the 0.5 entailment threshold are replaced with the first two sentences of the source abstract.",
        "- **Study types** are rule-based classifications from abstract text, not author-assigned MeSH terms.",
        "- **Keywords** are frequency-ranked tokens from the article title and abstract.",
    ]

    return "\n".join(lines)


report_md = generate_markdown_report(all_results, n_corpus_articles=len(articles))

display(Markdown(report_md[:4000] + "\n\n*... (truncated for preview)*"))

# Biomedical Literature Review Report

**Generated:** 2026-06-02 06:21 UTC  
**Data source:** PubMed Central Open Access (oa_comm)  
**Corpus size:** 100 articles  
**Retrieval model:** `sentence-transformers/all-MiniLM-L6-v2`  
**Summarization model:** `facebook/bart-large-cnn`  
**Verification model:** `facebook/bart-large-mnli`  

---

## Query: Adverse events with mRNA vaccines in pediatrics

*5 articles retrieved*

### 1. The Pregnancy, Arsenic, and Immune Response (PAIR) Study in rural northern Bangladesh

| Field | Value |
|---|---|
| Citation | [PMID:36756808](https://pubmed.ncbi.nlm.nih.gov/36756808/) |
| DOI | [10.1111/ppe.12949](https://doi.org/10.1111/ppe.12949) |
| Authors | Avolio L, Smith T, Navas‐Acien A, Kruczynski K, Pisanic N |
| Journal | Paediatric and Perinatal Epidemiology (2023) |
| Study type | `other` |
| Retrieval score | 0.3392 |
| Verification score | 0.9969 (Passed) |

**Summary:** The Pregnancy, Arsenic, and Immune Response (PAIR) Study was designed to assess whether arsenic exposure and micronutrient deficiencies alter maternal and newborn immunity and acute morbidity following maternal seasonal influenza vaccination during pregnancy. Of 784 women who enrolled, 736 (93.9%) delivered live births and 551 (70.3%) completed follow‐up visits

**Keywords:** `arsenic`, `women`, `maternal`, `water`, `pregnancy`

### 2. Rare and low-frequency coding genetic variants contribute to pediatric-onset multiple sclerosis

| Field | Value |
|---|---|
| Citation | [PMID:36755464](https://pubmed.ncbi.nlm.nih.gov/36755464/) |
| DOI | [10.1177/13524585221150736](https://doi.org/10.1177/13524585221150736) |
| Authors | Horton M, Shim J, Wallace A, Graves J, Aaen G |
| Journal | Multiple Sclerosis (Houndmills, Basingstoke, England) (2023) |
| Study type | `other` |
| Retrieval score | 0.2921 |
| Verification score | 0.9986 (Passed) |

**Summary:** Rare genetic variants are emerging as important contributors to the heritability of multiple sclerosis. Whether rare variants also contribute to pediatric-onset multiple sclerosis (POMS) is unknown. We analyzed DNA samples from 330 POMS cases and 306 controls for which Illumina ExomeChip genotypes were available.

**Keywords:** `rare`, `poms`, `variants`, `genes`, `onset`

### 3. Pediatric surgical site infections in 287 hospitals in the United States, 2015–2018

| Field | Value |
|---|---|
| Citation | [PMID:35801814](https://pubmed.ncbi.nlm.nih.gov/35801814/) |
| DOI | [10.1017/ice.2022.154](https://doi.org/10.1017/ice.2022.154) |
| Authors | Mathew R, Salinas J, Hsu H, Jin R, Rhee C |
| Journal | Infection Control and Hospital Epidemiology (2023) |
| Study type | `other` |
| Retrieval score | 0.2741 |
| Verification score | 0.9994 (Passed) |

**Summary:** Among 287 US hospitals reporting data between 2015 and 2018, annual pediatric surgical site infection (SSI) rates ranged from 0% for gallbladder to 10.4% for colon surgeries. Colon, spinal fusion, and small-bowel SSI rates did not decrease with greater surgical volumes.

**Keywords:** `surgical`, `rates`, `pediatric`, `site`, `hospitals`

### 4. Comparative transcriptomics from intestinal cells of permissive and non-permissive hosts during Ancylostoma ceylanicum infection reveals unique signatures of protection and host specificity

| Field | Value |
|---|---|
| Citation | [PMID:36883013](https://pubmed.ncbi.nlm.nih.gov/36883013/) |
| DOI | [10.1017/S0031182023000227](https://doi.org/10.1017/S0031182023000227) |
| Authors | Langeland A, Grill E, Shetty A, O'Halloran D, Hawdon J |
| Journal | Parasitology (2023) |
| Study type | `other` |
| Retrieval score | 0.2489 |
| Verification score | 0.9960 (Passed) |

**Summary:** Soil-transmitted nematodes (STNs) place a tremendous burden on health and economics. At least 1.5 billion people, or 24% of the population, are infected with at least 1 STN globally. What determines host specificity remains unanswered.

**Keywords:** `permissive`, `host`, `specificity`, `infection`, `prov

*... (truncated for preview)*

In [ ]:
import os

os.makedirs("outputs", exist_ok=True)

with open("outputs/literature_review_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)
print(f"Report saved: outputs/literature_review_report.md ({len(report_md):,} chars)")

with open("outputs/pipeline_results.json", "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, default=str)
print("Results saved: outputs/pipeline_results.json")

total_articles = sum(r["articles_retrieved"] for r in all_results.values())
total_passed   = sum(
    sum(1 for a in r["articles"] if a["verification_flag"] == "passed")
    for r in all_results.values()
)
print()
print("Pipeline summary")
print(f"  Queries run            : {len(all_results)}")
print(f"  Total articles output  : {total_articles}")
print(f"  Summaries verified     : {total_passed}/{total_articles}")

Report saved: outputs/literature_review_report.md (14,322 chars)
Results saved: outputs/pipeline_results.json

Pipeline summary
  Queries run            : 3
  Total articles output  : 15
  Summaries verified     : 14/15


In [ ]:
print("=" * 60)
print("PIPELINE ARCHITECTURE SUMMARY")
print("=" * 60)
print()
print("  [User Query]")
print("       |")
print("       v")
print("  RetrieverAgent")
print(f"    Model : {EMBEDDER_MODEL}")
print("    Index : cosine similarity over title+abstract corpus (384-dim)")
print("    Output: top-K articles ranked by relevance score")
print("       |")
print("       v")
print("  SummarizerAgent")
print(f"    Model : {SUMMARIZER_MODEL}")
print("    Output: 2-3 sentence abstractive summary + frequency-based keywords")
print("       |")
print("       v")
print("  VerifierAgent")
print(f"    Model : {VERIFIER_MODEL}")
print("    Method: NLI entailment (abstract as premise, summary as hypothesis)")
print("    Output: verified summary + entailment score + pass/fallback flag")
print("       |")
print("       v")
print("  Report Generator")
print("    Outputs: literature_review_report.md, pipeline_results.json")
print()
print("Design choices and tradeoffs are documented in README.md")

PIPELINE ARCHITECTURE SUMMARY

  [User Query]
       |
       v
  RetrieverAgent
    Model : sentence-transformers/all-MiniLM-L6-v2
    Index : cosine similarity over title+abstract corpus (384-dim)
    Output: top-K articles ranked by relevance score
       |
       v
  SummarizerAgent
    Model : facebook/bart-large-cnn
    Output: 2-3 sentence abstractive summary + frequency-based keywords
       |
       v
  VerifierAgent
    Model : facebook/bart-large-mnli
    Method: NLI entailment (abstract as premise, summary as hypothesis)
    Output: verified summary + entailment score + pass/fallback flag
       |
       v
  Report Generator
    Outputs: literature_review_report.md, pipeline_results.json

Design choices and tradeoffs are documented in README.md
